# Phutball PBT over GRPO (A100)

Runs `expert/grpo_pbt.py` from branch `expert-dagger`, starting from the self-play final
(`My Drive/phutball/selfplay/selfplay_21x15_final.pkl`). Logs, checkpoints and `control.json` live in
`My Drive/phutball/grpo/`. Run all; after a disconnect, run all again (it resumes from `grpo/state.pkl`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/phutball'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install mctx==0.0.6 2>&1 | tail -1   # keep Colab's JAX/Flax (the self-play run works on them)

In [ ]:
!mkdir -p /content/data && rsync -a --exclude '*.tmp*' $DRIVE/data/expert_data /content/data/
import os
INIT = f'{DRIVE}/selfplay/selfplay_21x15_final.pkl'
assert os.path.exists(INIT), 'the self-play run has not finished yet (no selfplay_21x15_final.pkl)'
RUN = f'{DRIVE}/grpo'; os.makedirs(RUN, exist_ok=True)
ARGS = f'--init {INIT} --puzzle-ref /content/data/expert_data/place_run/latest.pkl --run-dir {RUN} --data-root /content/data'
print(ARGS)

In [ ]:
# run loop: exit code 3 = reload requested (pull and relaunch), 0 = stop, other = crash (retry)
import subprocess, sys, time
while True:
    subprocess.run('git pull -q && git log --oneline -1', shell=True, cwd='/content/phutball-jax')
    p = subprocess.Popen(f'{sys.executable} -u -m expert.grpo_pbt {ARGS}', shell=True, cwd='/content/phutball-jax',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(f'{RUN}/stdout.log', 'a') as out:
        for line in p.stdout:
            print(line, end=''); out.write(line); out.flush()
    code = p.wait(); print('exit code', code)
    if code == 3: continue
    if code != 0:
        print('crashed; retrying in 60 s (see stdout.log)'); time.sleep(60); continue
    break